# Datamine REGMOD Process Example

This notebook demonstrates how to configure and run the **`regmod`** process wrapper in `dmstudio`.

## Process Description

## REGMOD

Produces a regular cell model from any type of Datamine ore body block model.

The parameters for the output regular cell model are taken from the prototype model &**IN1**. The model to be converted is &**IN2**. The regular model is output to &OUT.

The prototype model may be constructed by using the [PROTOM](<protom.md>) process. Any number of cells and any cell size may be chosen; **REGMOD** computes a weighted average of all input cells and sub-cells falling within the new cell size. However, it is better to choose a multiple or sub-multiple of the input cell size whenever possible, with the new origin chosen so that as many cell boundaries coincide between the old and new models as possible.

The output model will usually have larger cells than the input. However, **REGMOD** will also allow the input model to be divided into smaller cells, if required. Note however that values are always output for whole cells only; sub-cells will not exist in the output model.

The output model will contain two extra fields; **FILLVOL** , which is the total volume in the output cell filled by cells and sub-cells in the input model; and **VOIDVOL** , which is the total cell volume minus **FILLVOL** , or the total volume of the output cell which was not covered by cells and sub-cells.

Up to 25 numeric fields from the input model are averaged and placed in the output model. These fields are specified as field names * **F1** to * **F25**. Averaging is by volume weighting. There must be at least one numeric value field in the input model (* **F1**).

It should be noted that absent grade values are treated as ZERO when regularizing. For example if a cell is divided into two equal subcells of volume 500 each and the AU value in one subcell is 1, and - (absent data) in the other, then the regularised AU value over the parent cell is calculated as 0.5, the **FILLVOL** as 1000 and the **VOIDVOL** as 0. However, if the subcell with absent data is removed, leaving just one subcell, then the regularized **AU** value is 1, the **FILLVOL** is 500 and **VOIDVOL** is 500.

This treatment of absent data values has been designed specifically for calculating the revenue value of a cell in the regularized model, using a process such as [EXTRA](<extra.md>). For example if the revenue for a unit of **AU** is $100 then the dollar value is calculated as:

revenue = FILLVOL x grade x $value

* Case 1: revenue = 1000 x 0.5 x 100 = 50,000

* Case 2: revenue = 500 x 1.0 x 100 = 50,000

If **REGMOD** is used for other purposes then the implication of this treatment of absent data grade values should be considered carefully.

### Input Files:

* **in1** (Block Model Prototype):
  Input model prototype file, defining the new model origin, number of cells and cell
  sizes. This is typically set up by process **PROTOM** or create-model-prototype.
  Required=Yes

* **in2** (Block Model):
  Input model file for conversion. This must have the fields **XMORIG, YMORIG, ZMORIG,
  NX, NY, NZ** (implicit) and **IJK, XC, YC and ZC** (explicit). **XINC, YINC** and
  **ZINC** must exist as either explicit (sub-cells permitted) or implicit (no sub-cells).
  There must also be at least one explicit numeric data field, to be specified as **F1**.
  The records may be in any order, but speed is increased if they are in IJK order.
  Required=Yes

* **fieldlst** (Undefined):
  File to supply selected fields.
  Required=No

### Output Files:

* **out** (Block Model):
  Output model file. This will have the model parameters of the input prototype file on
  IN1 , and may contain up to twenty five averaged fields ( F1-F25). It will also contain
  the fields FILLVOL and VOIDVOL.
  Required=Yes

### Fields:

* **f1_f25** (Any : IN2):
  Explicit numeric fields to be averaged.
  Default=Undefined
  Required=No

* **fieldnam** (Character : FIELDLST):
  Field in **FIELDLST** to identify selected fields.
  Default=Undefined
  Required=No

### Parameters:

* **print**:
  >=2 display for each input cell or sub-cell that intersects with an output model cell;
  **IJK1,IJK,NUMMET,XC,YC,ZC,VOLP,VOLT,F1** [IJK of input and output cell, sub-cell no.,
  input cell centre, volume intersected, total volume to date in output cell, **F1**
  value] and for each output cell, **IJK, FILLVOL** and **VOIDVOL** values, and the **F1**
  value. (0).
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('regmod')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute regmod
print("Running regmod...")
dm_cmd.regmod(
    inmods_i=['optional'],  # required
    fieldlst_i='optional',  # required
    out_o='t_regmod_out',  # required
    # f1_f25_f='optional',  # optional
    # fieldnam_f='optional',  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("regmod execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_regmod_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")